# Experimentos de qualidade e velocidade da importância de features no Bankai

Este notebook investiga três formas de melhorar a importância nativa por ganho do Bankai, usando **crédito** e **fraude**: cenários em que o benchmark anterior mostrou mais espaço para melhorar. Ele compara estratégias de criação de bins, fração de features candidatas por split e quantidade de árvores.

O Bankai é instalado do **PyPI**; o notebook confirma que pacote e extensão nativa não vieram do checkout local. As configurações são comparadas com os mesmos dados, split e seed por repetição. O modo completo usa 500 mil linhas, 500 features, 10 repetições e um warmup por configuração. O modo leve reduz o custo e serve para validar execução; seus resultados não devem ser comparados aos do modo completo.

## 1. Instalar dependências

In [4]:
%pip install bankai-random-forest scikit-learn psutil threadpoolctl pandas numpy scipy

Se o Colab solicitar, reinicie o ambiente depois da instalação e execute as células novamente a partir do início.

## 2. Recursos da máquina

In [5]:
import os, platform, sys
import psutil
from threadpoolctl import threadpool_info

CPU_COUNT = os.cpu_count() or 1
PHYSICAL_CORES = psutil.cpu_count(logical=False) or CPU_COUNT
memory = psutil.virtual_memory()
print({
    'machine': platform.machine(), 'platform': platform.platform(),
    'python': sys.version, 'logical_cpus': CPU_COUNT,
    'physical_cores': PHYSICAL_CORES,
    'memory_total_gib': round(memory.total / 2**30, 2),
    'memory_available_gib': round(memory.available / 2**30, 2),
    'threadpools_before_import': threadpool_info(),
})
N_JOBS = CPU_COUNT

{'machine': 'x86_64', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'logical_cpus': 44, 'physical_cores': 22, 'memory_total_gib': 172.95, 'memory_available_gib': 167.51, 'threadpools_before_import': [{'user_api': 'blas', 'internal_api': 'openblas', 'num_threads': 44, 'prefix': 'libscipy_openblas', 'filepath': '/usr/local/lib/python3.13/dist-packages/numpy.libs/libscipy_openblas64_-ff651d7f.so', 'version': '0.3.27', 'threading_layer': 'pthreads', 'architecture': 'SkylakeX'}]}


## 3. Desenho e cenários

**Crédito (1,5% positivos):** representa inadimplência futura. Há sinais graduais de comprometimento e histórico, variáveis binárias e monetárias, uma relação não linear, uma interação e proxies correlacionadas. O tamanho de folha base é 500 e a profundidade máxima é 8.

**Fraude (0,2% positivos):** representa fraude confirmada em transações. Inclui valores extremos, limiares, variáveis binárias e interações, com somente cerca de mil positivos em 500 mil linhas. O tamanho de folha base é 750 e a profundidade máxima é 9.

Em ambos, as primeiras 24 colunas são informativas por construção; as demais são ruído. A prevalência é controlada calibrando o intercepto do sinal. São cenários sintéticos, úteis para comparação controlada, não dados reais de clientes.

## 4. Configuração, modo leve e persistência no Drive

In [6]:
from pathlib import Path
import json

LIGHT_MODE = False  # False: configuração completa de 500k linhas e 10 repetições
SAVE_TO_GOOGLE_DRIVE = True  # recomendado no Colab para retomar após desconexão
BASE_SEED = 20261011
N_WARMUP = 1
MAX_BINS = 63  # constante em todos os braços do experimento

if LIGHT_MODE:
    N_ROWS, N_FEATURES, N_INFORMATIVE = 20_000, 100, 24
    N_REPEATS, BASE_TREES, HIGH_TREES = 1, 20, 60
    SAMPLED_BIN_SIZES = (5_000, 10_000)
else:
    N_ROWS, N_FEATURES, N_INFORMATIVE = 500_000, 500, 24
    N_REPEATS, BASE_TREES, HIGH_TREES = 10, 100, 300
    SAMPLED_BIN_SIZES = (50_000, 100_000)

try:
    from google.colab import drive
    print("I'm running in Google Colab!")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    RESULT_DIR = Path('/content/drive/MyDrive/bankai_feature_importance_experiments')
else:
    RESULT_DIR = Path.cwd() / 'bankai_feature_importance_experiments'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODE_TAG = 'light' if LIGHT_MODE else 'full'
RAW_PATH = RESULT_DIR / f'runs-{MODE_TAG}.csv'
IMPORTANCE_PATH = RESULT_DIR / f'feature-ranks-{MODE_TAG}.csv'
SUMMARY_PATH = RESULT_DIR / f'summary-{MODE_TAG}.csv'
PAIRED_PATH = RESULT_DIR / f'paired-vs-baseline-{MODE_TAG}.csv'
STABILITY_PATH = RESULT_DIR / f'feature-stability-{MODE_TAG}.csv'

# Mantém aproximadamente a mesma fração de linhas por folha no modo leve.
LEAF_SCALE = N_ROWS / 500_000
SCENARIOS = {
    'credit_default': {'prevalence': 0.015, 'max_depth': 8, 'base_min_leaf': 500},
    'transaction_fraud': {'prevalence': 0.002, 'max_depth': 9, 'base_min_leaf': 750},
}

print({
    'light_mode': LIGHT_MODE, 'rows_per_scenario': N_ROWS,
    'features': N_FEATURES, 'informative_features': N_INFORMATIVE,
    'measured_repeats_per_configuration_scenario': N_REPEATS,
    'warmups_per_configuration_scenario': N_WARMUP,
    'baseline_trees': BASE_TREES, 'higher_tree_count': HIGH_TREES,
    'max_bins': MAX_BINS, 'n_jobs': N_JOBS, 'result_dir': str(RESULT_DIR),
})

Mounted at /content/drive
{'light_mode': False, 'rows_per_scenario': 500000, 'features': 500, 'informative_features': 24, 'measured_repeats_per_configuration_scenario': 10, 'warmups_per_configuration_scenario': 1, 'baseline_trees': 100, 'higher_tree_count': 300, 'max_bins': 63, 'n_jobs': 44, 'result_dir': '/content/drive/MyDrive/bankai_feature_importance_experiments'}


## 5. Braços experimentais

Cada grupo varia um fator por vez; os demais parâmetros ficam no baseline (100 árvores, `max_features="sqrt"`, `max_bins=63`, bootstrap com 80% das linhas, `class_weight="balanced"`, profundidade e folha definidos pelo cenário).

| Grupo | Comparações | O que responde |
|---|---|---|
| Binning | `exact_sort` (baseline), `exact_select`, `sampled_select` com 50k/100k linhas | Se a criação de bins pode ficar mais rápida mantendo ou preservando o ranking |
| Features por split | `sqrt` (baseline), `0.1`, `0.2` | Se mais candidatas ajudam a encontrar sinais fracos |
| Número de árvores | 100 (baseline), 300 | Se mais árvores estabilizam o ranking e justificam o custo |

`exact_select` usa a seleção dos quantis sobre todas as observações, sem ordenar integralmente cada coluna. `sampled_select` aproxima os quantis usando uma amostra; o tamanho da amostra muda entre modo leve e completo. `max_bins` permanece em 63 para isolar o efeito da estratégia. No modo leve, 100/300 árvores viram 20/60 e os tamanhos de amostra de bins diminuem.

O modo completo roda 7 configurações únicas × 2 cenários × 10 repetições, além de um warmup por configuração e cenário: **154 ajustes**. Memória necessária pode ultrapassar 2 GiB durante divisão dos dados; verifique os recursos acima.

## 6. Imports e confirmação de instalação via PyPI

In [7]:
import gc, math, time, warnings
from importlib import metadata
from scipy.special import expit
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from threadpoolctl import threadpool_limits
import bankai_random_forest
from bankai_random_forest import BankaiRandomForestClassifier

WORKDIR = Path.cwd().resolve()
REPO = WORKDIR.parent if WORKDIR.name == 'benchmarks' else WORKDIR
package_file = Path(bankai_random_forest.__file__).resolve()
extension_file = Path(bankai_random_forest._core.__file__).resolve()
dist = metadata.distribution('bankai-random-forest')
direct_url = dist.read_text('direct_url.json')
if package_file.is_relative_to(REPO) or extension_file.is_relative_to(REPO):
    raise RuntimeError(f'Bankai foi importado do checkout local, esperado PyPI: {package_file}')
if direct_url and json.loads(direct_url).get('dir_info', {}).get('editable', False):
    raise RuntimeError('Bankai está instalado em modo editable; este benchmark requer o pacote do PyPI.')
params = BankaiRandomForestClassifier().get_params(deep=False)
required = {'binning_strategy', 'bin_sample_size'}
if not required.issubset(params):
    raise RuntimeError(f'A versão instalada do Bankai não expõe {sorted(required)}; atualize a versão PyPI usada neste notebook.')
print({
    'bankai_version': dist.version,
    'bankai_package': str(package_file),
    'bankai_native_extension': str(extension_file),
    'scikit_learn': metadata.version('scikit-learn'),
    'numpy': metadata.version('numpy'),
})

{'bankai_version': '0.1.0a10', 'bankai_package': '/usr/local/lib/python3.13/dist-packages/bankai_random_forest/__init__.py', 'bankai_native_extension': '/usr/local/lib/python3.13/dist-packages/bankai_random_forest/_core.abi3.so', 'scikit_learn': '1.9.1', 'numpy': '2.1.3'}


## 7. Geradores reproduzíveis dos datasets

In [8]:
def calibrate_intercept(signal, prevalence):
    signal = (signal - signal.mean()) / max(signal.std(), 1e-8)
    lo, hi = -30.0, 30.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if expit(signal + mid).mean() > prevalence:
            hi = mid
        else:
            lo = mid
    return signal + (lo + hi) / 2

def make_dataset(scenario, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(N_ROWS, N_FEATURES)).astype(np.float32)
    X[:, 4:8] = rng.binomial(1, [0.12, 0.08, 0.18, 0.35], size=(N_ROWS, 4)).astype(np.float32)
    X[:, 8:12] = np.log1p(rng.lognormal(mean=2.0, sigma=0.8, size=(N_ROWS, 4))).astype(np.float32)
    X[:, 8:12] = (X[:, 8:12] - X[:, 8:12].mean(axis=0)) / (X[:, 8:12].std(axis=0) + 1e-6)
    X[:, 16:20] = 0.75 * X[:, 0:4] + 0.66 * rng.normal(size=(N_ROWS, 4)).astype(np.float32)
    z = X
    if scenario == 'credit_default':
        signal = (1.0*z[:,0] + .85*z[:,1] + .65*z[:,2] + .5*z[:,3]
                  + .8*(z[:,4] > .5) + .7*(z[:,5] > .5) + .5*z[:,6] - .4*z[:,7]
                  + .65*z[:,8] - .55*z[:,9] + .45*z[:,10] + .35*z[:,11]
                  + .7*np.maximum(z[:,12]-.8, 0) + .6*(z[:,13]**2-1)
                  + .65*z[:,14]*z[:,15] + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'transaction_fraud':
        signal = (1.0*z[:,0] + .8*np.maximum(z[:,1]-1, 0) + .7*(z[:,2] > 1.2)
                  + .9*z[:,3]*z[:,4] + .7*(z[:,5] > .5) + .5*z[:,6]
                  + .8*np.abs(z[:,7]) + .55*z[:,8] + .5*z[:,9]
                  + .65*(z[:,10]**2-1) + .7*z[:,11]*z[:,12]
                  + .5*z[:,13] + .4*z[:,14] + .35*z[:,15]
                  + .18*z[:,16:24].sum(axis=1))
    else:
        raise ValueError(f'Cenário desconhecido: {scenario}')
    logits = calibrate_intercept(signal, SCENARIOS[scenario]['prevalence'])
    y = rng.binomial(1, expit(logits)).astype(np.int8)
    informative = np.arange(N_FEATURES) < N_INFORMATIVE
    return X, y, informative

print({name: cfg for name, cfg in SCENARIOS.items()})

{'credit_default': {'prevalence': 0.015, 'max_depth': 8, 'base_min_leaf': 500}, 'transaction_fraud': {'prevalence': 0.002, 'max_depth': 9, 'base_min_leaf': 750}}


## 8. Configurações Bankai e métricas

In [9]:
def scenario_min_leaf(scenario):
    return max(5, int(round(SCENARIOS[scenario]['base_min_leaf'] * LEAF_SCALE)))

CONFIGS = {
    'baseline': {'experiment': 'baseline', 'n_estimators': BASE_TREES,
                 'max_features': 'sqrt', 'binning_strategy': 'exact_sort', 'bin_sample_size': 200_000},
    'exact_select': {'experiment': 'binning', 'n_estimators': BASE_TREES,
                     'max_features': 'sqrt', 'binning_strategy': 'exact_select', 'bin_sample_size': 200_000},
    'sampled_select_50k': {'experiment': 'binning', 'n_estimators': BASE_TREES,
                           'max_features': 'sqrt', 'binning_strategy': 'sampled_select', 'bin_sample_size': min(SAMPLED_BIN_SIZES[0], N_ROWS)},
    'sampled_select_100k': {'experiment': 'binning', 'n_estimators': BASE_TREES,
                            'max_features': 'sqrt', 'binning_strategy': 'sampled_select', 'bin_sample_size': min(SAMPLED_BIN_SIZES[1], N_ROWS)},
    'max_features_0.1': {'experiment': 'max_features', 'n_estimators': BASE_TREES,
                         'max_features': 0.1, 'binning_strategy': 'exact_sort', 'bin_sample_size': 200_000},
    'max_features_0.2': {'experiment': 'max_features', 'n_estimators': BASE_TREES,
                         'max_features': 0.2, 'binning_strategy': 'exact_sort', 'bin_sample_size': 200_000},
    'trees_high': {'experiment': 'tree_count', 'n_estimators': HIGH_TREES,
                   'max_features': 'sqrt', 'binning_strategy': 'exact_sort', 'bin_sample_size': 200_000},
}
EXPERIMENTS = {
    'binning': ['baseline', 'exact_select', 'sampled_select_50k', 'sampled_select_100k'],
    'max_features': ['baseline', 'max_features_0.1', 'max_features_0.2'],
    'tree_count': ['baseline', 'trees_high'],
}

def make_bankai(config_id, scenario, seed):
    cfg = CONFIGS[config_id]
    return BankaiRandomForestClassifier(
        n_estimators=cfg['n_estimators'], max_depth=SCENARIOS[scenario]['max_depth'],
        min_samples_leaf=scenario_min_leaf(scenario), max_features=cfg['max_features'],
        max_samples=0.80, bootstrap=True, max_bins=MAX_BINS,
        binning_strategy=cfg['binning_strategy'], bin_sample_size=cfg['bin_sample_size'],
        importance_type='gain', class_weight='balanced', random_state=int(seed % (2**31 - 1)),
        n_jobs=N_JOBS,
    )

TOP_K = N_INFORMATIVE

def evaluate_one(X, y, informative, scenario, config_id, repeat, data_seed, model_seed):
    split_seed = int((data_seed + 73) % (2**31 - 1))
    train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.20,
                                           stratify=y, random_state=split_seed)
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    model = make_bankai(config_id, scenario, model_seed)
    start = time.perf_counter()
    with threadpool_limits(limits=N_JOBS):
        model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    with threadpool_limits(limits=N_JOBS):
        probability = model.predict_proba(X_test)[:, 1]
    prediction_seconds = time.perf_counter() - start
    importance = np.nan_to_num(np.asarray(model.feature_importances_, dtype=np.float64),
                               nan=0.0, posinf=0.0, neginf=0.0)
    order = np.argsort(-importance, kind='stable')
    rank = np.empty_like(order)
    rank[order] = np.arange(1, len(order) + 1)
    top = order[:TOP_K]
    true_positives = int(informative[top].sum())
    cfg = CONFIGS[config_id]
    row = {
        'scenario': scenario, 'experiment': cfg['experiment'], 'config_id': config_id,
        'repeat': repeat, 'data_seed': data_seed, 'model_seed': model_seed,
        'fit_seconds': fit_seconds, 'prediction_seconds': prediction_seconds,
        'feature_average_precision': float(average_precision_score(informative.astype(int), importance)),
        'recall_at_24': true_positives / int(informative.sum()),
        'roc_auc': float(roc_auc_score(y_test, probability)),
        'pr_auc': float(average_precision_score(y_test, probability)),
        'n_estimators': cfg['n_estimators'], 'max_features': str(cfg['max_features']),
        'binning_strategy': cfg['binning_strategy'], 'bin_sample_size': cfg['bin_sample_size'],
        'max_bins': MAX_BINS, 'min_samples_leaf': scenario_min_leaf(scenario),
        'max_depth': SCENARIOS[scenario]['max_depth'], 'n_jobs': N_JOBS,
    }
    per_feature = pd.DataFrame({
        'scenario': scenario, 'config_id': config_id, 'repeat': repeat,
        'data_seed': data_seed, 'feature_index': np.arange(len(importance)),
        'importance': importance, 'rank': rank,
        'is_informative': informative.astype(bool),
        'in_top_24': rank <= TOP_K,
    })
    del model, X_train, X_test, y_train, y_test, probability
    return row, per_feature

## 9. Warmup, execuções pareadas e retomada

Cada repetição gera cada cenário uma vez; todas as configurações usam exatamente a mesma matriz, rótulos, split e seed do estimador. Cada ajuste é salvo em CSV logo após concluir. No Colab, o Drive mantém checkpoints em desconexões; reexecute a célula para continuar. Uma assinatura impede misturar resultados de configurações diferentes na mesma execução.

In [10]:
signature_payload = {
    'light_mode': LIGHT_MODE, 'rows': N_ROWS, 'features': N_FEATURES,
    'repeats': N_REPEATS, 'base_trees': BASE_TREES, 'high_trees': HIGH_TREES,
    'base_seed': BASE_SEED, 'max_bins': MAX_BINS, 'n_jobs': N_JOBS,
    'configs': CONFIGS, 'scenarios': SCENARIOS,
}
RUN_SIGNATURE = json.dumps(signature_payload, sort_keys=True)

records = pd.read_csv(RAW_PATH).to_dict('records') if RAW_PATH.exists() else []
feature_records = pd.read_csv(IMPORTANCE_PATH).to_dict('records') if IMPORTANCE_PATH.exists() else []
if any(row.get('run_signature') != RUN_SIGNATURE for row in records):
    raise RuntimeError('O CSV raw tem outra configuração. Escolha outra pasta de resultados ou remova o arquivo antigo.')
done = {(row['scenario'], row['config_id'], int(row['repeat'])) for row in records}

def checkpoint(frame, path):
    temp_path = path.with_name(path.name + '.tmp')
    frame.to_csv(temp_path, index=False)
    temp_path.replace(path)

for scenario_index, scenario in enumerate(SCENARIOS):
    print(f'\n=== Warmup: {scenario} ===', flush=True)
    warm_seed = BASE_SEED + scenario_index * 1_000_000 + 777
    X_warm, y_warm, _ = make_dataset(scenario, warm_seed)
    train_idx, _ = train_test_split(np.arange(len(y_warm)), test_size=0.20,
                                    stratify=y_warm, random_state=(warm_seed + 73) % (2**31 - 1))
    for config_index, config_id in enumerate(CONFIGS):
        model_seed = BASE_SEED + scenario_index * 1_000_000 + 800 + config_index * 101
        warm_model = make_bankai(config_id, scenario, model_seed)
        with threadpool_limits(limits=N_JOBS):
            warm_model.fit(X_warm[train_idx], y_warm[train_idx])
        del warm_model
        gc.collect()
        print('warmup complete:', config_id, flush=True)
    del X_warm, y_warm, train_idx
    gc.collect()

    for repeat in range(N_REPEATS):
        if all((scenario, config_id, repeat) in done for config_id in CONFIGS):
            continue
        data_seed = BASE_SEED + scenario_index * 1_000_000 + repeat * 10_000 + 500
        model_seed = BASE_SEED + scenario_index * 1_000_000 + repeat * 10_000 + 900
        X, y, informative = make_dataset(scenario, data_seed)
        for config_id in CONFIGS:
            key = (scenario, config_id, repeat)
            if key in done:
                continue
            print(f'{scenario} | {repeat + 1}/{N_REPEATS} | {config_id}', flush=True)
            row, feature_frame = evaluate_one(X, y, informative, scenario, config_id,
                                              repeat, data_seed, model_seed)
            row['run_signature'] = RUN_SIGNATURE
            feature_frame['run_signature'] = RUN_SIGNATURE
            records.append(row)
            feature_records.extend(feature_frame.to_dict('records'))
            records = (pd.DataFrame(records)
                       .drop_duplicates(['scenario', 'config_id', 'repeat'], keep='last')
                       .to_dict('records'))
            feature_records = (pd.DataFrame(feature_records)
                               .drop_duplicates(['scenario', 'config_id', 'repeat', 'feature_index'], keep='last')
                               .to_dict('records'))
            checkpoint(pd.DataFrame(feature_records), IMPORTANCE_PATH)
            checkpoint(pd.DataFrame(records), RAW_PATH)
            done.add(key)
            print({k: row[k] for k in ('fit_seconds', 'feature_average_precision',
                                       'recall_at_24', 'pr_auc')}, flush=True)
        del X, y, informative
        gc.collect()
print('Raw runs:', RAW_PATH)
print('Feature rankings:', IMPORTANCE_PATH)


=== Warmup: credit_default ===
warmup complete: baseline
warmup complete: exact_select
warmup complete: sampled_select_50k
warmup complete: sampled_select_100k
warmup complete: max_features_0.1
warmup complete: max_features_0.2
warmup complete: trees_high
credit_default | 1/10 | baseline
{'fit_seconds': 4.592635450999296, 'feature_average_precision': 0.8292294602110525, 'recall_at_24': 0.7916666666666666, 'pr_auc': 0.04286693422003468}
credit_default | 1/10 | exact_select
{'fit_seconds': 4.474162579999756, 'feature_average_precision': 0.7747492349604418, 'recall_at_24': 0.75, 'pr_auc': 0.038722935484987064}
credit_default | 1/10 | sampled_select_50k
{'fit_seconds': 4.274125481999363, 'feature_average_precision': 0.8187766901149814, 'recall_at_24': 0.7916666666666666, 'pr_auc': 0.03891060908792109}
credit_default | 1/10 | sampled_select_100k
{'fit_seconds': 4.270575423000082, 'feature_average_precision': 0.7693166820844896, 'recall_at_24': 0.75, 'pr_auc': 0.041748732912344405}
credit_d

## 10. Resumo, comparações pareadas e estabilidade

In [11]:
raw = pd.read_csv(RAW_PATH)
feature_ranks = pd.read_csv(IMPORTANCE_PATH)

# Média do Jaccard entre conjuntos top-24 das repetições: indica consistência
# da seleção entre seeds; valores altos não substituem a recuperação da verdade.
def mean_pairwise_jaccard(group):
    sets = [set(part.nsmallest(TOP_K, 'rank')['feature_index']) for _, part in group.groupby('repeat')]
    if len(sets) < 2:
        return np.nan
    scores = []
    for i in range(len(sets)):
        for j in range(i + 1, len(sets)):
            union = sets[i] | sets[j]
            scores.append(len(sets[i] & sets[j]) / len(union) if union else 1.0)
    return float(np.mean(scores))

stability = (feature_ranks.groupby(['scenario', 'config_id'], sort=False)
             .apply(mean_pairwise_jaccard, include_groups=False)
             .rename('top24_pairwise_jaccard').reset_index())
summary = (raw.groupby(['scenario', 'config_id'], sort=False)
          .agg(runs=('repeat', 'nunique'), experiment=('experiment', 'first'),
               n_estimators=('n_estimators', 'first'), max_features=('max_features', 'first'),
               binning_strategy=('binning_strategy', 'first'), bin_sample_size=('bin_sample_size', 'first'),
               fit_median_s=('fit_seconds', 'median'),
               fit_iqr_s=('fit_seconds', lambda x: x.quantile(.75) - x.quantile(.25)),
               feature_ap_median=('feature_average_precision', 'median'),
               recall_at_24_median=('recall_at_24', 'median'),
               roc_auc_median=('roc_auc', 'median'), pr_auc_median=('pr_auc', 'median'))
          .reset_index().merge(stability, on=['scenario', 'config_id'], how='left'))
summary.to_csv(SUMMARY_PATH, index=False)

paired_rows = []
metric_cols = ['fit_seconds', 'feature_average_precision', 'recall_at_24', 'pr_auc', 'roc_auc']
for experiment, config_ids in EXPERIMENTS.items():
    candidates = [c for c in config_ids if c != 'baseline']
    for scenario in SCENARIOS:
        base = raw[(raw.scenario == scenario) & (raw.config_id == 'baseline')]
        for candidate in candidates:
            other = raw[(raw.scenario == scenario) & (raw.config_id == candidate)]
            paired = base[['repeat', *metric_cols]].merge(other[['repeat', *metric_cols]],
                         on='repeat', suffixes=('_baseline', '_candidate'), validate='one_to_one')
            item = {'scenario': scenario, 'experiment': experiment, 'config_id': candidate,
                    'paired_runs': len(paired)}
            for metric in metric_cols:
                delta = paired[f'{metric}_candidate'] - paired[f'{metric}_baseline']
                item[f'median_delta_{metric}'] = float(delta.median()) if len(delta) else np.nan
            item['median_fit_ratio_candidate_over_baseline'] = float(
                (paired['fit_seconds_candidate'] / paired['fit_seconds_baseline']).median()) if len(paired) else np.nan
            item['fraction_faster'] = float((paired.fit_seconds_candidate < paired.fit_seconds_baseline).mean()) if len(paired) else np.nan
            item['fraction_higher_feature_ap'] = float((paired.feature_average_precision_candidate > paired.feature_average_precision_baseline).mean()) if len(paired) else np.nan
            item['fraction_higher_recall_at_24'] = float((paired.recall_at_24_candidate > paired.recall_at_24_baseline).mean()) if len(paired) else np.nan
            paired_rows.append(item)
paired_summary = pd.DataFrame(paired_rows)
paired_summary.to_csv(PAIRED_PATH, index=False)

feature_stability = (feature_ranks.groupby(['scenario', 'config_id', 'feature_index'], as_index=False)
    .agg(is_informative=('is_informative', 'first'), median_rank=('rank', 'median'),
         top24_frequency=('in_top_24', 'mean'), median_importance=('importance', 'median')))
feature_stability.to_csv(STABILITY_PATH, index=False)

print('Medianas por configuração:')
display(summary.sort_values(['scenario', 'experiment', 'fit_median_s']))
print('Diferenças pareadas versus baseline (mesmos dados, split e seeds):')
display(paired_summary)
print('Arquivos salvos em:', RESULT_DIR)

Medianas por configuração:


,scenario,config_id,runs,experiment,n_estimators,max_features,binning_strategy,bin_sample_size,fit_median_s,fit_iqr_s,feature_ap_median,recall_at_24_median,roc_auc_median,pr_auc_median,top24_pairwise_jaccard
0,credit_default,baseline,10,baseline,100,sqrt,exact_sort,200000,4.540789,0.072661,0.822702,0.791667,0.710371,0.038889,0.586342
2,credit_default,sampled_select_50k,10,binning,100,sqrt,sampled_select,50000,4.208483,0.049562,0.756682,0.729167,0.708106,0.037829,0.532312
3,credit_default,sampled_select_100k,10,binning,100,sqrt,sampled_select,100000,4.257988,0.054415,0.791505,0.750000,0.710804,0.038235,0.562475
1,credit_default,exact_select,10,binning,100,sqrt,exact_select,200000,4.493447,0.038148,0.778122,0.750000,0.709891,0.038537,0.542667
4,credit_default,max_features_0.1,10,max_features,100,0.1,exact_sort,200000,4.778506,0.069185,0.800830,0.750000,0.715289,0.039709,0.560111
5,credit_default,max_features_0.2,10,max_features,100,0.2,exact_sort,200000,4.856223,0.101866,0.756770,0.708333,0.714379,0.039499,0.519622
6,credit_default,trees_high,10,tree_count,300,sqrt,exact_sort,200000,11.238483,0.130400,0.852093,0.812500,0.717282,0.041125,0.633533
7,transaction_fraud,baseline,10,baseline,100,sqrt,exact_sort,200000,4.336044,0.115682,0.600304,0.583333,0.683920,0.005123,0.329564
10,transaction_fraud,sampled_select_100k,10,binning,100,sqrt,sampled_select,100000,4.020588,0.101557,0.594792,0.541667,0.676145,0.005111,0.316511
9,transaction_fraud,sampled_select_50k,10,binning,100,sqrt,sampled_select,50000,4.055050,0.078799,0.600986,0.583333,0.675840,0.005681,0.325224


Diferenças pareadas versus baseline (mesmos dados, split e seeds):


,scenario,experiment,config_id,paired_runs,median_delta_fit_seconds,median_delta_feature_average_precision,median_delta_recall_at_24,median_delta_pr_auc,median_delta_roc_auc,median_fit_ratio_candidate_over_baseline,fraction_faster,fraction_higher_feature_ap,fraction_higher_recall_at_24
0,credit_default,binning,exact_select,10,-0.069535,-0.051715,-0.041667,-0.000695,-0.002905,0.984714,0.7,0.1,0.1
1,credit_default,binning,sampled_select_50k,10,-0.305720,-0.053212,-0.041667,-0.001144,-0.003155,0.932962,1.0,0.1,0.1
2,credit_default,binning,sampled_select_100k,10,-0.265273,-0.049707,-0.041667,-0.000871,-0.002334,0.941524,1.0,0.1,0.1
3,transaction_fraud,binning,exact_select,10,-0.093066,-0.018952,-0.020833,-0.000360,-0.004645,0.978387,0.9,0.4,0.1
4,transaction_fraud,binning,sampled_select_50k,10,-0.281266,-0.006679,0.000000,-0.000260,-0.012952,0.935563,1.0,0.4,0.3
5,transaction_fraud,binning,sampled_select_100k,10,-0.303656,-0.018614,-0.020833,-0.000243,-0.015743,0.930374,1.0,0.4,0.2
6,credit_default,max_features,max_features_0.1,10,0.224001,-0.028009,0.000000,0.000451,0.005262,1.049382,0.0,0.2,0.1
7,credit_default,max_features,max_features_0.2,10,0.340695,-0.053238,-0.041667,0.000426,0.005066,1.074428,0.0,0.0,0.0
8,transaction_fraud,max_features,max_features_0.1,10,-0.005514,-0.014601,-0.020833,0.000713,0.008925,0.998721,0.6,0.4,0.1
9,transaction_fraud,max_features,max_features_0.2,10,-0.036996,-0.025227,-0.041667,0.001675,0.017282,0.991532,0.7,0.4,0.3


Arquivos salvos em: /content/drive/MyDrive/bankai_feature_importance_experiments


## 11. Como interpretar

- **Binning:** `exact_select` mais rápido com feature AP/recall parecidos ao baseline apoia a troca com baixo risco. `sampled_select` só compensa se o ganho de tempo vier sem queda consistente na qualidade pareada.
- **Features por split:** procure aumento de feature AP e recall@24 em crédito/fraude. Confira o custo de tempo e o `top24_pairwise_jaccard`; qualidade maior com ranking instável pode exigir mais árvores ou seeds.
- **Árvores:** compare 100 vs 300 (ou 20 vs 60 no modo leve). Considere o custo adicional junto da melhoria de qualidade/estabilidade.
- `feature_average_precision` e `recall_at_24` usam a verdade sintética de 24 features informativas. `PR-AUC` e `ROC-AUC` medem a predição do rótulo em holdout e contextualizam o ajuste.
- Use medianas e diferenças pareadas; uma execução pequena pode inverter rankings por acaso. Resultados do modo leve validam o fluxo e não substituem o modo completo.

A importância por ganho continua sendo uma medida interna do modelo, não causalidade. Este notebook não calcula permutation importance.